Перед запуском данного борда необходимо сделать следующее:

0. Сделать копию борда через меню "Файл" / "File".
1. Выбрать пункт "Среда выполнения" или "Runtime" и выберите GPU в качестве аппаратного ускорителя в пункте меню "Сменить среду выполнения".


# Инструменты для работы с языком

... или зачем нужна предобработка.

Раньше мы смотрели на светлую сторону анализа данных - построение моделей. Теперь попробуем глубже посмотреть на часть про предобработку данных. Задача предобработки особенно актуальна, если мы имеем дело с текстами.

## Задача: классификация твитов по тональности

У нас есть выборка из твитов.
Нам известна эмоциональная окраска каждого твита из выборки: положительная или отрицательная. Задача состоит в построении модели, которая по тексту твита предсказывает его эмоциональную окраску.

Классификацию по тональности используют в рекомендательных системах, чтобы понять, понравилось ли людям кафе, кино, etc.

Скачиваем выборку ([источник](http://study.mokoron.com/)): [положительные](https://raw.githubusercontent.com/Gavroshe/RuTweetCorp/master/positive.csv), [отрицательные]( https://raw.githubusercontent.com/Gavroshe/RuTweetCorp/master/negative.csv).

In [2]:
import os
import urllib.request

# переключаем рабочую директорию в папку с ноутбуком
try:
    notebook_dir = os.path.dirname(os.path.abspath(__file__))
except NameError:
    notebook_dir = r'c:\Users\Aleksandr Riabkov\Desktop\Itmo.py\ITMO.py\second_semester\LR7. NLP'
os.chdir(notebook_dir)
print('Рабочая директория:', os.getcwd())

for fname, url in [
    ('positive.csv', 'https://raw.githubusercontent.com/Gavroshe/RuTweetCorp/master/positive.csv'),
    ('negative.csv', 'https://raw.githubusercontent.com/Gavroshe/RuTweetCorp/master/negative.csv'),
]:
    if not os.path.exists(fname):
        print(f'Скачивание {fname}...')
        urllib.request.urlretrieve(url, fname)
    print(f'{fname}: OK')

Рабочая директория: c:\Users\Aleksandr Riabkov\Desktop\Itmo.py\ITMO.py\second_semester\LR7. NLP
positive.csv: OK
negative.csv: OK


In [3]:
import os
import pandas as pd
import numpy as np

from sklearn.metrics import *
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline

In [4]:
pd.read_csv('positive.csv', sep=';', header=None).head()

,0,1,2,3,4,5,6,7,8,9,10,11
0,408906692374446080,1386325927,pleease_shut_up,"@first_timee хоть я и школота, но поверь, у на...",1,0,0,0,7569,62,61,0
1,408906692693221377,1386325927,alinakirpicheva,"Да, все-таки он немного похож на него. Но мой ...",1,0,0,0,11825,59,31,2
2,408906695083954177,1386325927,EvgeshaRe,RT @KatiaCheh: Ну ты идиотка) я испугалась за ...,1,0,1,0,1273,26,27,0
3,408906695356973056,1386325927,ikonnikova_21,"RT @digger2912: ""Кто то в углу сидит и погибае...",1,0,1,0,1549,19,17,0
4,408906761416867842,1386325943,JumpyAlex,@irina_dyshkant Вот что значит страшилка :D\nН...,1,0,0,0,597,16,23,1


In [5]:
pd.read_csv('negative.csv', sep=';', header=None).tail()

,0,1,2,3,4,5,6,7,8,9,10,11
111918,425138243257253888,1390195830,Yanch_96,Но не каждый хочет что то исправлять:( http://...,-1,0,0,0,1138,32,46,0
111919,425138339503943682,1390195853,tkit_on,скучаю так :-( только @taaannyaaa вправляет мо...,-1,0,0,0,4822,38,32,0
111920,425138437684215808,1390195876,ckooker1,"Вот и в школу, в говно это идти уже надо(",-1,0,0,1,165,13,16,0
111921,425138490452344832,1390195889,LisaBeroud,"RT @_Them__: @LisaBeroud Тауриэль, не грусти :...",-1,0,1,0,2516,187,265,0
111922,425138595251625984,1390195914,sukapavlov,Такси везет меня на работу. Раздумываю приплат...,-1,0,0,0,7778,146,66,5


Откроем файлы и создадим массив из текстов и правильных меток для твитов.
Сначала идут положительные твиты, потом отрицательные.

In [6]:
# загружаем положительные твиты
# TODO #2
positive = pd.read_csv('positive.csv', sep=';', usecols=[3], names=['text'])
positive['label'] = ['positive'] * len(positive)
# загружаем отрицательные твиты
# TODO #3
negative = pd.read_csv('negative.csv', sep=';', usecols=[3], names=['text'])
negative['label'] = ['negative'] * len(negative)
# соединяем вместе
# TODO #4
df = pd.concat([positive, negative])


In [7]:
df

,text,label
0,"@first_timee хоть я и школота, но поверь, у на...",positive
1,"Да, все-таки он немного похож на него. Но мой ...",positive
2,RT @KatiaCheh: Ну ты идиотка) я испугалась за ...,positive
3,"RT @digger2912: ""Кто то в углу сидит и погибае...",positive
4,@irina_dyshkant Вот что значит страшилка :D\nН...,positive
...,...,...
111918,Но не каждый хочет что то исправлять:( http://...,negative
111919,скучаю так :-( только @taaannyaaa вправляет мо...,negative
111920,"Вот и в школу, в говно это идти уже надо(",negative
111921,"RT @_Them__: @LisaBeroud Тауриэль, не грусти :...",negative


Посмотрим на полученные данные:

In [8]:
df.sample(5, random_state=40)

,text,label
15931,RT @Blawar_1337: Теперь у нас с @Wake_UA появи...,positive
59532,с днём рождения зайка*))) ухх погуляем мы сего...,positive
47185,RT @Shumkova0406199: @ann_safina Вов вов вов А...,negative
42002,"Надо выдернуть звуковую дорожку из ""Доктора Ка...",positive
109035,@_hassliebe_ может все таки на этой неделе вер...,negative


Разбиваем данные на обучающую и тестовую выборки с помощью функции ```train_test_split()``` из **sklearn**:


In [9]:
x_train, x_test, y_train, y_test = train_test_split(df.text, df.label)


print(x_train.shape, x_test.shape, y_train.shape, y_test.shape)

(170125,) (56709,) (170125,) (56709,)


In [10]:
y_train[:10]

42689    positive
3875     positive
54296    positive
36628    positive
6758     positive
18068    positive
39285    positive
11480    positive
62056    negative
82631    positive
Name: label, dtype: str

In [11]:
y_train.value_counts()

label
positive    85952
negative    84173
Name: count, dtype: int64

## Baseline: классификация необработанных n-грамм

* Сейчас мы попробуем получить преобразование предложений в численный вектор, с которым может работать стандартный алгоритм машинного обучения, такой как логистическая регрессия.
* Для этого нам понадобится познакомиться с понятием n-gram - самых мелких элементов предложения, с которыми можно работать.
* Подсчитав количество этих n-грам в предложениях, мы получим искомые численные представления.

## Что такое n-граммы:

Самые мелкие структуры языка, с которыми мы работаем, называются **n-граммами**.
У n-граммы есть параметр n - количество слов, которые попадают в такое представление текста.
* Если n = 1 - то мы смотрим на то, сколько раз каждое слово встретилось в тексте. Получаем _униграммы_
* Если n = 2 - то мы смотрим на то, сколько раз каждая пара подряд идущих слов, встретилась в тексте. Получаем _биграммы_

**Функция** для работы с n-граммами реализована в библиотке **nltk** (Natural Language ToolKit), импортируем эту функцию:

In [12]:
from nltk import ngrams

Прежде чем получать n-граммы, нужно разделить предложение на отдельные слова.  Для этого используем метод ```split()```.

In [13]:
sentence = 'Если б мне платили каждый раз'.split()
sentence

['Если', 'б', 'мне', 'платили', 'каждый', 'раз']

Чтобы получить n-грамму для такой последовательности, используем функцию ```ngrams()```.

На вход передается два параметра:
* лист с разделенным на отдельные слова предложением (у нас он хранится в переменной ```sent```);
* параметр n, определяющий, какой тип n-грамм мы хотим получить.


Чтобы полученный объект отобразить, делаем из него ```list```.

In [14]:
list(ngrams(sentence, 1)) # униграммы

[('Если',), ('б',), ('мне',), ('платили',), ('каждый',), ('раз',)]

Аналогично мы можем получить биграммы - для этого заменяем параметр **n** в функции **ngrams** с 1 на 2.

In [15]:
list(ngrams(sentence, 2)) # биграммы

[('Если', 'б'),
 ('б', 'мне'),
 ('мне', 'платили'),
 ('платили', 'каждый'),
 ('каждый', 'раз')]

In [16]:
list(ngrams(sentence, 3)) # триграммы

[('Если', 'б', 'мне'),
 ('б', 'мне', 'платили'),
 ('мне', 'платили', 'каждый'),
 ('платили', 'каждый', 'раз')]

In [17]:
list(ngrams(sentence, 5)) # ... пентаграммы?

[('Если', 'б', 'мне', 'платили', 'каждый'),
 ('б', 'мне', 'платили', 'каждый', 'раз')]

### Векторизаторы

Векторизатор преобразует слово или набор слов в числовой вектор, понятный алгоритму машинного обучения, который привык работать с числовыми табличными данными.

Ниже - пример преобразования слов в двумерных вектор, каждому слову соответствует точка на плоскости.

<a href="https://drive.google.com/uc?id=1ukv-FTj0jeVdcgVlOaNBocUfNuYGGVZg
" target="_blank"><img src="https://drive.google.com/uc?id=1ukv-FTj0jeVdcgVlOaNBocUfNuYGGVZg"
alt="IMAGE ALT TEXT HERE" width="600" border="0" /></a>

На начальном этапе нам будет достаточно тех инструментов, которые уже есть в знакомой нам библиотеке **sklearn**.

In [18]:
from sklearn.linear_model import LogisticRegression # можно заменить на любимый классификатор
from sklearn.feature_extraction.text import CountVectorizer # модель "мешка слов", см. далее

Самый простой способ извлечь признаки из текстовых данных -- векторизаторы: `CountVectorizer` и `TfidfVectorizer`

Объект `CountVectorizer` делает следующую вещь:
* строит для каждого документа (каждой пришедшей ему строки) вектор размерности `n`, где `n` -- количество слов или n-грам во всём корпусе
* заполняет каждый i-тый элемент количеством вхождений слова в данный документ

<a href="https://drive.google.com/uc?id=1ukv-FTj0jeVdcgVlOaNBocUfNuYGGVZg
" target="_blank"><img src="https://drive.google.com/uc?id=1jHmkrGZTMawM46Yzxh243Ur1y5pYKzrl"
alt="IMAGE ALT TEXT HERE" width="600" border="0" /></a>

На рисунке пример векторизации для униграмм, но можно использовать любые n-граммы. Для этого у объекта ```CountVectorizer()``` есть параметр **ngram_range**, который отвечает за то, какие n-граммы мы используем в качестве признаов:<br/>
ngram_range=(1, 1) -- униграммы<br/>
ngram_range=(3, 3) -- триграммы<br/>
ngram_range=(1, 3) -- униграммы, биграммы и триграммы.

<a href="https://drive.google.com/uc?id=1ODNVK0fdLTX4nv6ob55ciUe37d1pio-D" target="_blank"><img src="https://drive.google.com/uc?id=1ODNVK0fdLTX4nv6ob55ciUe37d1pio-D"
alt="IMAGE ALT TEXT HERE" width="800" border="0" /></a>

Инициализируем ```CountVectorizer()```, указав в качестве признаков униграммы:

In [19]:
vectorizer = CountVectorizer(ngram_range=(1, 2))

После инициализации _vectorizer_ можно обучить на наших данных.

Для обучения используем обучающую выборку ```x_train```, но в отличие от классификатора мы используем метод ```fit_transform()```: сначала обучаем наш векторизатор, а потом сразу применяем его к нашему набору данных. Это похоже на то, как мы работали с label encoderом и one-hot-encoderом.


In [20]:
# TODO #8 - обучаем векторизатор на обучающей выборке и сразу применяем
vectorized_x_train = vectorizer.fit_transform(x_train)

In [21]:
vectorized_x_train = vectorizer.fit_transform(x_train)

Так как результат не зависит от порядка слов в текстах, то говорят, что такая модель представления текстов в виде векторов получается из *гипотезы представления текста как мешка слов*

В vectorizer.vocabulary_ лежит словарь, отображение слов в их индексы:

In [22]:
list(vectorizer.vocabulary_.items())[:10]

[('определение', 799386),
 ('местоположения', 656398),
 ('подсказывает', 863912),
 ('вы', 386377),
 ('россии', 974671),
 ('зачем', 524716),
 ('знать', 532631),
 ('точнее', 1116280),
 ('россия', 975076),
 ('же', 489489)]

В нашей выборке 170125 текстов (твитов), в них встречается 243760 разных слов.

In [23]:
vectorized_x_train.shape

(170125, 1247468)

Так как теперь у нас есть **численное представление** и набор входных признаков, то мы можем обучить модель логистической регрессии (или любую другую из тех, на которые мы смотрели раньше, например, случайный лес)

In [24]:
clf = LogisticRegression(random_state=42, max_iter=1000)
clf.fit(vectorized_x_train, y_train)

,"penalty penalty: {'l1', 'l2', 'elasticnet', None}, default='l2'Specify the norm of the penalty:- `None`: no penalty is added;- `'l2'`: add a L2 penalty term and it is the default choice;- `'l1'`: add a L1 penalty term;- `'elasticnet'`: both L1 and L2 penalty terms are added... warning:: Some penalties may not work with some solvers. See the parameter `solver` below, to know the compatibility between the penalty and solver... versionadded:: 0.19 l1 penalty with SAGA solver (allowing 'multinomial' + L1).. deprecated:: 1.8 `penalty` was deprecated in version 1.8 and will be removed in 1.10. Use `l1_ratio` instead. `l1_ratio=0` for `penalty='l2'`, `l1_ratio=1` for `penalty='l1'` and `l1_ratio` set to any float between 0 and 1 for `'penalty='elasticnet'`.",'deprecated'
,"C C: float, default=1.0Inverse of regularization strength; must be a positive float.Like in support vector machines, smaller values specify strongerregularization. `C=np.inf` results in unpenalized logistic regression.For a visual example on the effect of tuning the `C` parameterwith an L1 penalty, see::ref:`sphx_glr_auto_examples_linear_model_plot_logistic_path.py`.",1.0
,"l1_ratio l1_ratio: float, default=0.0The Elastic-Net mixing parameter, with `0 <= l1_ratio <= 1`. Setting`l1_ratio=1` gives a pure L1-penalty, setting `l1_ratio=0` a pure L2-penalty.Any value between 0 and 1 gives an Elastic-Net penalty of the form`l1_ratio * L1 + (1 - l1_ratio) * L2`... warning:: Certain values of `l1_ratio`, i.e. some penalties, may not work with some solvers. See the parameter `solver` below, to know the compatibility between the penalty and solver... versionchanged:: 1.8 Default value changed from None to 0.0... deprecated:: 1.8 `None` is deprecated and will be removed in version 1.10. Always use `l1_ratio` to specify the penalty type.",0.0
,"dual dual: bool, default=FalseDual (constrained) or primal (regularized, see also:ref:`this equation `) formulation. Dual formulationis only implemented for l2 penalty with liblinear solver. Prefer `dual=False`when n_samples > n_features.",False
,"tol tol: float, default=1e-4Tolerance for stopping criteria.",0.0001
,"fit_intercept fit_intercept: bool, default=TrueSpecifies if a constant (a.k.a. bias or intercept) should beadded to the decision function.",True
,"intercept_scaling intercept_scaling: float, default=1Useful only when the solver `liblinear` is usedand `self.fit_intercept` is set to `True`. In this case, `x` becomes`[x, self.intercept_scaling]`,i.e. a ""synthetic"" feature with constant value equal to`intercept_scaling` is appended to the instance vector.The intercept becomes``intercept_scaling * synthetic_feature_weight``... note:: The synthetic feature weight is subject to L1 or L2 regularization as all other features. To lessen the effect of regularization on synthetic feature weight (and therefore on the intercept) `intercept_scaling` has to be increased.",1
,"class_weight class_weight: dict or 'balanced', default=NoneWeights associated with classes in the form ``{class_label: weight}``.If not given, all classes are supposed to have weight one.The ""balanced"" mode uses the values of y to automatically adjustweights inversely proportional to class frequencies in the input dataas ``n_samples / (n_classes * np.bincount(y))``.Note that these weights will be multiplied with sample_weight (passedthrough the fit method) if sample_weight is specified... versionadded:: 0.17 *class_weight='balanced'*",None
,"random_state random_state: int, RandomState instance, default=NoneUsed when ``solver`` == 'sag', 'saga' or 'liblinear' to shuffle thedata. See :term:`Glossary ` for details.",42
,"solver solver: {'lbfgs', 'liblinear', 'newton-cg', 'newton-cholesky', 'sag', 'saga'}, default='lbfgs'Algorithm to use in the optimization problem. Default is 'lbfgs'.To choose a solver, you might want to consider the following aspects:- 'lbfgs' is a good default solver because it works reasonably well for a wide class of problems.- For :term:`multi

С тестовыми данными нужно проделать то же самое, что и с данными для обучения: сделать из текстов вектора, которые можно передавать в классификатор для прогноза класса объекта.

У нас уже есть обученный векторизатор ```vectorizer```, поэтому используем метод ```transform()``` (просто применить его), а не ```fit_transform``` (обучить и применить).

In [25]:
vectorized_x_test = vectorizer.transform(x_test)

Как раньше, для получения прогноза у обученного классификатора используем метод ```predict()```.

С помощью функции ```classification_report()```, которая считает сразу несколько метрик качества классификации, посмотрим на то, насколько хорошо мы предсказываем положительную или отрицательную тональность твита .

для range (1,3)
 negative       0.78      0.78      0.78     28001
positive       0.79      0.78      0.78     28708

для (1,1)
negative       0.76      0.77      0.76     28001
positive       0.77      0.76      0.77     28708

для (1,2)
negative       0.78      0.78      0.78     28001
positive       0.78      0.78      0.78     28708

In [26]:
pred = clf.predict(vectorized_x_test)

print(classification_report(y_test, pred))

              precision    recall  f1-score   support

    negative       0.77      0.78      0.78     27750
    positive       0.79      0.78      0.78     28959

    accuracy                           0.78     56709
   macro avg       0.78      0.78      0.78     56709
weighted avg       0.78      0.78      0.78     56709



## Бонус*: триграммы

Попробуем сделать то же самое, используя в качестве признаков триграммы:

In [27]:
# TODO #12

# инициализируем векторайзер
vectorizer_3 = CountVectorizer(ngram_range=(3,3))
# обучаем его и сразу применяем к x_train
vectorized_x_train_3 = vectorizer_3.fit_transform(x_train)
# инициализируем и обучаем классификатор
clf = LogisticRegression(random_state=42, max_iter=1000)
clf.fit(vectorized_x_train_3, y_train)
# применяем обученный векторизатор к тестовым данным
vectorized_x_test_3 = vectorizer_3.transform(x_test)
# получаем предсказания и выводим информацию о качестве
pred = clf.predict(vectorized_x_test_3)
print(classification_report(y_test, pred))

              precision    recall  f1-score   support

    negative       0.71      0.47      0.57     27750
    positive       0.62      0.82      0.70     28959

    accuracy                           0.65     56709
   macro avg       0.66      0.64      0.63     56709
weighted avg       0.66      0.65      0.64     56709



Как вы думаете, почему в результатах теперь такой разброс по сравнению с униграммами?

**Ответ:** твиты короткие, поэтому конкретные тройки слов встречаются крайне редко. Большинство триграмм из тестовой выборки вообще не попадались при обучении — у модели нет информации для их классификации. В такой ситуации модель "угадывает" один класс чаще другого (здесь — positive), отсюда высокий recall у positive (0.82) и низкий у negative (0.47). С униграммами такой проблемы нет: отдельные слова встречаются в обучении гораздо чаще.

## Бонус**: TF-IDF векторизация

`TfidfVectorizer` делает то же, что и `CountVectorizer`, но в качестве значений выдает **tf-idf** каждого слова.

Как считается tf-idf:

**TF (term frequency)** – относительная частотность слова в документе:
$$ TF(t,d) = \frac{n_{t}}{\sum_k n_{k}} $$

**IDF (inverse document frequency)** – обратная частота документов, в которых есть это слово:
$$ IDF(t, D) = \mbox{log} \frac{|D|}{|{d : t \in d}|} $$

Перемножаем их:
$$TFIDF(t, d, D) = TF(t,d) \times IDF(i, D)$$

Сакральный смысл: если слово часто встречается в одном документе, но в целом по корпусу встречается в небольшом
количестве документов, у него высокий TF-IDF.

In [28]:
from sklearn.feature_extraction.text import TfidfVectorizer

Действуем аналогично, как с ```CountVectorizer()```:

In [29]:
# TODO #13

# инициализируем векторизатор, в качестве переменных используем униграммы
tfidfvectorizer = TfidfVectorizer(ngram_range=(1,5))
# обучаем его и сразу применяем к x_train
tfidf_vectorized_x_train = tfidfvectorizer.fit_transform(x_train)
# инициализируем и обучаем классификатор
clf = LogisticRegression(random_state=42, max_iter=1000)
# применяем обученный векторизатор к тестовым данным
clf.fit(tfidf_vectorized_x_train, y_train)

# получаем предсказания и выводим информацию о качестве
tfidf_vectorized_x_test  = tfidfvectorizer.transform(x_test)

# # получаем предсказания и выводим информацию о качестве
pred = clf.predict(tfidf_vectorized_x_test)
print(classification_report(y_test, pred))

              precision    recall  f1-score   support

    negative       0.73      0.78      0.75     27750
    positive       0.77      0.72      0.75     28959

    accuracy                           0.75     56709
   macro avg       0.75      0.75      0.75     56709
weighted avg       0.75      0.75      0.75     56709



In [30]:
# TODO #14

# инициализируем векторизатор с пентаграммами
tfidf_penta = TfidfVectorizer(ngram_range=(5, 5))
# обучаем его и сразу применяем к x_train
vectorized_train_penta = tfidf_penta.fit_transform(x_train)
# инициализируем и обучаем классификатор
clf_penta = LogisticRegression(random_state=42, max_iter=1000)
clf_penta.fit(vectorized_train_penta, y_train)
# применяем обученный векторизатор к тестовым данным
vectorized_test_penta = tfidf_penta.transform(x_test)
# получаем предсказания и выводим информацию о качестве
pred_penta = clf_penta.predict(vectorized_test_penta)
print(classification_report(y_test, pred_penta))

              precision    recall  f1-score   support

    negative       0.94      0.11      0.20     27750
    positive       0.54      0.99      0.70     28959

    accuracy                           0.56     56709
   macro avg       0.74      0.55      0.45     56709
weighted avg       0.74      0.56      0.45     56709



## Токенизация

Токенизировать - значит, поделить текст на части: слова, ключевые слова, фразы, символы и т.д., иными словами **токены**.

Самый наивный способ токенизировать текст - разделить с помощью функции `split()`. Но `split` упускает очень много всего, например, не отделяет пунктуацию от слов. Кроме этого, есть ещё много менее тривиальных проблем, поэтому лучше использовать готовые токенизаторы.

In [31]:
import nltk # уже знакомая нам библиотека nltk
from nltk.tokenize import word_tokenize # готовый токенизатор библиотеки nltk

Чтобы использовать токенизатор ```word_tokenize```, нужно сначала скачать данные для nltk о пунктуации и стоп-словах. Это просто требование nltk, поэтому, особо не задумываясь, запустите следующую ячейку:  

In [32]:
nltk.download('punkt_tab')
nltk.download('stopwords')

[nltk_data] Downloading package punkt_tab to C:\Users\Aleksandr
[nltk_data]     Riabkov\AppData\Roaming\nltk_data...
[nltk_data]   Unzipping tokenizers\punkt_tab.zip.
[nltk_data] Downloading package stopwords to C:\Users\Aleksandr
[nltk_data]     Riabkov\AppData\Roaming\nltk_data...
[nltk_data]   Unzipping corpora\stopwords.zip.


True

Применим токенизацию:

In [33]:
example = 'Но не каждый хочет что-то исправлять:('
word_tokenize(example)

['Но', 'не', 'каждый', 'хочет', 'что-то', 'исправлять', ':', '(']

Если использовать просто ```split()```, то грустный смайлик :( не отделяется от слова "исправлять":

In [34]:
example.split()

['Но', 'не', 'каждый', 'хочет', 'что-то', 'исправлять:(']

В nltk вообще есть довольно много токенизаторов:

In [35]:
from nltk import tokenize
dir(tokenize)[:16]

['BlanklineTokenizer',
 'LegalitySyllableTokenizer',
 'LineTokenizer',
 'MWETokenizer',
 'NLTKWordTokenizer',
 'PunktSentenceTokenizer',
 'PunktTokenizer',
 'RegexpTokenizer',
 'ReppTokenizer',
 'SExprTokenizer',
 'SpaceTokenizer',
 'StanfordSegmenter',
 'SyllableTokenizer',
 'TabTokenizer',
 'TextTilingTokenizer',
 'ToktokTokenizer']

Они умеют выдавать индексы в строке для начала и конца каждого слова-токена:

In [36]:
wh_tok = tokenize.WhitespaceTokenizer()
list(wh_tok.span_tokenize(example))

[(0, 2), (3, 5), (6, 12), (13, 18), (19, 25), (26, 38)]

Некторые токенизаторы ведут себя специфично:

In [37]:
tokenize.TreebankWordTokenizer().tokenize("don't stop me")

['do', "n't", 'stop', 'me']

А некоторые -- вообще не для текста на естественном языке:

In [38]:
tokenize.SExprTokenizer().tokenize("(a (b c)) d e (f)")

['(a (b c))', 'd', 'e', '(f)']

**Правильный токенизатор подбирается исходя из требований задачи!**

## Стоп-слова и пунктуация

**Стоп-слова** - это слова, которые часто встречаются практически в любом тексте и ничего интересного не говорят о конретном документе. Для модели это просто шум. А шум нужно убирать. По аналогичной причине убирают и пунктуацию.

In [39]:
# импортируем стоп-слова из библиотеки nltk
from nltk.corpus import stopwords

# посмотрим на стоп-слова для русского языка
print(stopwords.words('russian'))

['и', 'в', 'во', 'не', 'что', 'он', 'на', 'я', 'с', 'со', 'как', 'а', 'то', 'все', 'она', 'так', 'его', 'но', 'да', 'ты', 'к', 'у', 'же', 'вы', 'за', 'бы', 'по', 'только', 'ее', 'мне', 'было', 'вот', 'от', 'меня', 'еще', 'нет', 'о', 'из', 'ему', 'теперь', 'когда', 'даже', 'ну', 'вдруг', 'ли', 'если', 'уже', 'или', 'ни', 'быть', 'был', 'него', 'до', 'вас', 'нибудь', 'опять', 'уж', 'вам', 'ведь', 'там', 'потом', 'себя', 'ничего', 'ей', 'может', 'они', 'тут', 'где', 'есть', 'надо', 'ней', 'для', 'мы', 'тебя', 'их', 'чем', 'была', 'сам', 'чтоб', 'без', 'будто', 'чего', 'раз', 'тоже', 'себе', 'под', 'будет', 'ж', 'тогда', 'кто', 'этот', 'того', 'потому', 'этого', 'какой', 'совсем', 'ним', 'здесь', 'этом', 'один', 'почти', 'мой', 'тем', 'чтобы', 'нее', 'сейчас', 'были', 'куда', 'зачем', 'всех', 'никогда', 'можно', 'при', 'наконец', 'два', 'об', 'другой', 'хоть', 'после', 'над', 'больше', 'тот', 'через', 'эти', 'нас', 'про', 'всего', 'них', 'какая', 'много', 'разве', 'три', 'эту', 'моя', 'впр

*Знаки* пунктуации лучше импортировать из модуля **String**. В нем хранятся различные наборы констант для работы со строками (пунктуация, алфавит и др.).

In [40]:
from string import punctuation
punctuation

'!"#$%&\'()*+,-./:;<=>?@[\\]^_`{|}~'

Объединим стоп-слова и знаки пунктуации вместе и запишем в переменную ```noise```:

In [41]:
noise = stopwords.words('russian') + list(punctuation)

Теперь нужно обучать нашу модель с учетом новых знаний про токенизацию и стоп-слова.

Для этого мы можем собрать новый векторизатор, передав ему на вход:
* какие n-граммы нам нужны, параметр **ngram_range**;
* какой токенизатор мы используем, параметр **tokenizer**;
* какие у нас стоп-слова, параметр **stop_words**.

*Напоминание:* мы используем готовый токенизатор ```word_tokenize```, а стоп-слова хранятся в переменной ```noise```

In [54]:
# инициализируем умный векторайзер
# TODO #16 с word_tokenize
smart_tokenizer = CountVectorizer(ngram_range=(1, 3), tokenizer=word_tokenize,
                                  stop_words=noise)

In [55]:
# TODO #17

# обучаем его и сразу применяем к x_train
smart_vectorized_x_train = smart_tokenizer.fit_transform(x_train)
# инициализируем и обучаем классификатор
clf_smart = LogisticRegression(random_state=42, max_iter=1000)
clf_smart.fit(smart_vectorized_x_train, y_train)
# применяем обученный векторайзер к тестовым данным
smart_vectorized_x_test = smart_tokenizer.transform(x_test)
# получаем предсказания и выводим информацию о качестве
pred_smart = clf_smart.predict(smart_vectorized_x_test)
print(classification_report(y_test, pred_smart))

c:\Users\Aleksandr Riabkov\Desktop\Itmo.py\ITMO.py\second_semester\LR7. NLP\venv\Lib\site-packages\sklearn\feature_extraction\text.py:526: UserWarning: The parameter 'token_pattern' will not be used since 'tokenizer' is not None'
  warnings.warn(
c:\Users\Aleksandr Riabkov\Desktop\Itmo.py\ITMO.py\second_semester\LR7. NLP\venv\Lib\site-packages\sklearn\feature_extraction\text.py:411: UserWarning: Your stop_words may be inconsistent with your preprocessing. Tokenizing the stop words generated tokens ['``'] not in stop_words.
  warnings.warn(


              precision    recall  f1-score   support

    negative       0.76      0.81      0.79     27750
    positive       0.81      0.76      0.78     28959

    accuracy                           0.79     56709
   macro avg       0.79      0.79      0.79     56709
weighted avg       0.79      0.79      0.79     56709



Получилось лучше: accuracy выше, а также заметно подрос recall у негативного класса.

Что ещё можно сделать?

## Бонус*: Лемматизация

**Лемматизация** – это сведение разных форм одного слова к начальной форме – **лемме**. Почему это хорошо?
* Во-первых, естественно рассматривать как отдельный признак каждое *слово*, а не каждую его отдельную форму.
* Во-вторых, некоторые стоп-слова стоят только в начальной форме, и без лематизации выкидываем мы только её.

Для русского есть хороших лемматизатор pymorphy.

### [Pymorphy](http://pymorphy2.readthedocs.io/en/latest/)
Это модуль на питоне, довольно быстрый и с кучей функций.

In [56]:
# устанавливаем pymorphy3
!pip install pymorphy3

В pymorphy2 для морфологического анализа слов есть ```MorphAnalyzer()```:

In [45]:
from pymorphy3 import MorphAnalyzer
# создадим объект pymorphy3_analyzer импортированного класса

# TODO #18
pymorphy3_analyzer = MorphAnalyzer()

pymorphy3 работает с отдельными словами. Если дать ему на вход предложение - он его просто не лемматизирует, т.к. не понимает:

In [57]:
sent = ['Если', 'б', 'мне', 'платили', 'каждый', 'раз']
sent

['Если', 'б', 'мне', 'платили', 'каждый', 'раз']

Лемматизируем слово "платили" из предложения ```sent``` с помощью метода ```parse()```:

In [58]:
ana = pymorphy3_analyzer.parse(sent[3])
ana

[Parse(word='платили', tag=OpencorporaTag('VERB,impf,tran plur,past,indc'), normal_form='платить', score=1.0, methods_stack=((DictionaryAnalyzer(), 'платили', 2471, 10),))]

Выведем его нормальную форму:

In [59]:
ana[0].normal_form

'платить'

## О важности эксплоративного анализа

Но иногда пунктуация бывает и не шумом - главное отталкиваться от задачи. Что будет если вообще не убирать пунктуацию?

In [60]:
# TODO #19

# инициализируем умный векторайзер stop-words НЕ ИСПОЛЬЗУЕМ!
no_stop_vectorizer = CountVectorizer(ngram_range=(1, 1), tokenizer=word_tokenize)
# обучаем его и сразу применяем к x_train
no_stop_vectorized_x_train = no_stop_vectorizer.fit_transform(x_train)
# инициализируем и обучаем классификатор
clf_no_stop = LogisticRegression(random_state=42, max_iter=1000)
clf_no_stop.fit(no_stop_vectorized_x_train, y_train)
# применяем обученный векторайзер к тестовым данным
no_stop_vectorized_x_test = no_stop_vectorizer.transform(x_test)
# получаем предсказания и выводим информацию о качестве
pred_no_stop = clf_no_stop.predict(no_stop_vectorized_x_test)
print(classification_report(y_test, pred_no_stop))

c:\Users\Aleksandr Riabkov\Desktop\Itmo.py\ITMO.py\second_semester\LR7. NLP\venv\Lib\site-packages\sklearn\feature_extraction\text.py:526: UserWarning: The parameter 'token_pattern' will not be used since 'tokenizer' is not None'
  warnings.warn(


              precision    recall  f1-score   support

    negative       1.00      1.00      1.00     27750
    positive       1.00      1.00      1.00     28959

    accuracy                           1.00     56709
   macro avg       1.00      1.00      1.00     56709
weighted avg       1.00      1.00      1.00     56709



Шок! Стоило оставить пунктуацию -- и все метрики равны 1. Как это получилось? Среди неё были очень значимые токены (как вы думаете, какие?).

Посмотрим, как один из супер-значительных токенов справится с классификацией безо всякого машинного обучения:

In [61]:
# TODO #20

# данные размечены по смайликам: positive.csv - твиты с :), negative.csv - с :(
# поэтому :( является идеальным предиктором негативного класса
smiley_pred = x_test.apply(lambda t: 'negative' if ':(' in t or ':-(' in t else 'positive')
print(classification_report(y_test, smiley_pred))

              precision    recall  f1-score   support

    negative       1.00      0.35      0.52     27750
    positive       0.62      1.00      0.76     28959

    accuracy                           0.68     56709
   macro avg       0.81      0.67      0.64     56709
weighted avg       0.80      0.68      0.64     56709



## Символьные n-граммы

Теперь в качестве фичей используем, например, униграммы символов. Для этого необходимо установить в ```CountVectorizer()``` параметр ```analyzer = 'char'```, то есть анализировать символы.

In [62]:
# TODO #21

# инициализируем векторайзер для символов
char_vectorizer = CountVectorizer(analyzer='char', ngram_range=(1, 1))
# обучаем его и сразу применяем к x_train
char_vectorized_x_train = char_vectorizer.fit_transform(x_train)
# инициализируем и обучаем классификатор
clf_char = LogisticRegression(random_state=42, max_iter=1000)
clf_char.fit(char_vectorized_x_train, y_train)
# применяем обученный векторайзер к тестовым данным
char_vectorized_x_test = char_vectorizer.transform(x_test)
# получаем предсказания и выводим информацию о качестве
pred_char = clf_char.predict(char_vectorized_x_test)
print(classification_report(y_test, pred_char))

              precision    recall  f1-score   support

    negative       1.00      0.99      0.99     27750
    positive       0.99      1.00      1.00     28959

    accuracy                           0.99     56709
   macro avg       1.00      0.99      0.99     56709
weighted avg       0.99      0.99      0.99     56709



Из предыдущего раздела уже понятно, почему на этих данных точность равна 1.

Символьные n-граммы используются, например, для задачи определения языка. Ещё одна замечательная особенность признаков-символов - для них не нужна токенизация и лемматизация, можно использовать такой подход для языков, у которых нет готовых анализаторов.

# Самостоятельная работа

1. Изучите материал, представленный в борде.
2. Выполните все ячейки и получите результаты.
3. Приведите результаты таблицы classification_report в под этим заданием для модели LogisticRegression
4. Примените 2 альтернативных использованному алгоритму для решения задачи классификации (для примера XGBClassifier и еще какой-то один) и получите результаты в таблице classification_report
5. Для XGBClassifier вам потребуется задать параметры
```learning_rate=0.1, n_estimators=1000, max_depth=5, min_child_weight=3, gamma=0.2, subsample=0.6, colsample_bytree=1.0, objective='binary:logistic', nthread=4, scale_pos_weight=1, seed=27```

6. В разделе TF-IDF векторизация по аналогии с униграммами и пентаграммами вычислите classification_report для биграмм, триграмм опубликуйте результаты в отчете и укажите изменилась ли точность f1-score при их использовании по сравнению с униграммами и пентаграммами.

## Результаты LogisticRegression (CountVectorizer, уни+биграммы)

Результаты из ячейки выше (baseline, `ngram_range=(1, 2)`):

```
              precision    recall  f1-score   support

    negative       0.77      0.78      0.78     27750
    positive       0.79      0.78      0.78     28959

    accuracy                           0.78     56709
   macro avg       0.78      0.78      0.78     56709
weighted avg       0.78      0.78      0.78     56709
```

## TF-IDF: биграммы и триграммы

In [52]:
# TF-IDF биграммы
tfidf_bi = TfidfVectorizer(ngram_range=(2, 2))
tfidf_bi_train = tfidf_bi.fit_transform(x_train)
clf_bi = LogisticRegression(random_state=42, max_iter=1000)
clf_bi.fit(tfidf_bi_train, y_train)
tfidf_bi_test = tfidf_bi.transform(x_test)
pred_bi = clf_bi.predict(tfidf_bi_test)
print("TF-IDF биграммы:")
print(classification_report(y_test, pred_bi))

# TF-IDF триграммы
tfidf_tri = TfidfVectorizer(ngram_range=(3, 3))
tfidf_tri_train = tfidf_tri.fit_transform(x_train)
clf_tri = LogisticRegression(random_state=42, max_iter=1000)
clf_tri.fit(tfidf_tri_train, y_train)
tfidf_tri_test = tfidf_tri.transform(x_test)
pred_tri = clf_tri.predict(tfidf_tri_test)
print("TF-IDF триграммы:")
print(classification_report(y_test, pred_tri))

TF-IDF биграммы:
              precision    recall  f1-score   support

    negative       0.72      0.67      0.69     27750
    positive       0.70      0.75      0.73     28959

    accuracy                           0.71     56709
   macro avg       0.71      0.71      0.71     56709
weighted avg       0.71      0.71      0.71     56709

TF-IDF триграммы:
              precision    recall  f1-score   support

    negative       0.72      0.46      0.56     27750
    positive       0.62      0.83      0.71     28959

    accuracy                           0.65     56709
   macro avg       0.67      0.64      0.63     56709
weighted avg       0.67      0.65      0.63     56709



## Альтернативные классификаторы

### XGBClassifier

In [64]:
from xgboost import XGBClassifier
from sklearn.preprocessing import LabelEncoder

# CountVectorizer с униграммами для XGB и LinearSVC
vec_base = CountVectorizer(ngram_range=(1, 1))
X_train_base = vec_base.fit_transform(x_train)
X_test_base = vec_base.transform(x_test)

# LabelEncoder для XGBClassifier (требует числовые метки)
le = LabelEncoder()
y_train_enc = le.fit_transform(y_train)
y_test_enc = le.transform(y_test)

xgb = XGBClassifier(
    learning_rate=0.1, n_estimators=1000, max_depth=5,
    min_child_weight=3, gamma=0.2, subsample=0.6,
    colsample_bytree=1.0, objective='binary:logistic',
    nthread=4, scale_pos_weight=1, seed=27
)
xgb.fit(X_train_base, y_train_enc)
pred_xgb = xgb.predict(X_test_base)
print("XGBClassifier:")
print(classification_report(y_test_enc, pred_xgb, target_names=le.classes_))

XGBClassifier:
              precision    recall  f1-score   support

    negative       0.74      0.68      0.71     27750
    positive       0.71      0.77      0.74     28959

    accuracy                           0.72     56709
   macro avg       0.72      0.72      0.72     56709
weighted avg       0.72      0.72      0.72     56709



### LinearSVC

In [65]:
from sklearn.svm import LinearSVC

# используем ту же векторизацию X_train_base / X_test_base
lsvc = LinearSVC(random_state=42, max_iter=2000)
lsvc.fit(X_train_base, y_train)
pred_lsvc = lsvc.predict(X_test_base)
print("LinearSVC:")
print(classification_report(y_test, pred_lsvc))

c:\Users\Aleksandr Riabkov\Desktop\Itmo.py\ITMO.py\second_semester\LR7. NLP\venv\Lib\site-packages\sklearn\svm\_base.py:1258: ConvergenceWarning: Liblinear failed to converge, increase the number of iterations.
  warnings.warn(


LinearSVC:
              precision    recall  f1-score   support

    negative       0.73      0.78      0.75     27750
    positive       0.77      0.72      0.75     28959

    accuracy                           0.75     56709
   macro avg       0.75      0.75      0.75     56709
weighted avg       0.75      0.75      0.75     56709



## Выводы по TF-IDF n-граммам

| Векторизатор | f1 negative | f1 positive | accuracy |
|---|---|---|---|
| TF-IDF уни-пентагр (1,5) | 0.75 | 0.75 | 0.75 |
| TF-IDF биграммы (2,2) | 0.69 | 0.73 | 0.71 |
| TF-IDF триграммы (3,3) | 0.56 | 0.71 | 0.65 |
| TF-IDF пентаграммы (5,5) | 0.20 | 0.70 | 0.56 |

TF-IDF (1,5) дал лучший f1-score среди n-граммных вариантов. С ростом n качество падает: биграммы хуже, триграммы ещё хуже, пентаграммы — сильно хуже из-за разреженности матрицы. Отдельные твиты слишком короткие, чтобы дать модели достаточно примеров длинных n-грамм для обобщения.

## Результаты альтернативных классификаторов (CountVectorizer, уни)

| Классификатор | f1 negative | f1 positive | accuracy |
|---|---|---|---|
| LogisticRegression | 0.78 | 0.78 | 0.78 |
| XGBClassifier | 0.71 | 0.74 | 0.72 |
| LinearSVC | 0.75 | 0.75 | 0.75 |

LogisticRegression показала лучший результат. XGBClassifier уступает: на разреженных текстовых данных 1000 деревьев склонны к переобучению без тщательной настройки.